# Day 28 — Data Storytelling & Communicating Insights
**30-Day Data Analytics Cookbook | From Foundations to Real-World Impact**

**Goal:** Translate municipal grievance data into a concise, evidence-based management briefing with charts, KPIs, and a measurable action plan.

All data in this notebook are **synthetic illustrations**, not actual GCC records.

**Learning objectives**
1. Calculate service delivery KPIs and communicate their limitations.
2. Select and annotate charts that support a specific decision.
3. Analyze complaint categories and zones to prioritize investigation.
4. Generate an executive summary with specific next steps and monitoring metrics.

> Data analytics is not about dashboards. It is about decisions.

## 1. Setup and illustrative data
Install dependencies if necessary: `pip install pandas numpy matplotlib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

months = pd.date_range('2026-01-01', periods=6, freq='MS')
monthly = pd.DataFrame({
    'month': months,
    'received': [1200, 1350, 1280, 1500, 1420, 1600],
    'resolved': [1140, 1260, 1220, 1350, 1300, 1400]
})
monthly['monthly_gap'] = monthly['received'] - monthly['resolved']
monthly['resolution_ratio_pct'] = monthly['resolved'] / monthly['received'] * 100
monthly['intake_growth_pct'] = monthly['received'].pct_change() * 100
monthly['cumulative_net_inflow'] = monthly['monthly_gap'].cumsum()
assert (monthly['resolved'] <= monthly['received']).all()
display(monthly.round(2))

### Metric definitions and caveats
- **Monthly resolution ratio** = cases resolved during the month ÷ cases received during the month. This is a *throughput ratio*, not a cohort closure rate: resolved cases may have been received earlier.
- **Monthly gap** = received − resolved. A positive gap represents net inflow into the unresolved workload, **assuming comparable scope and no other adjustments**; it is not necessarily the actual end-of-month backlog.
- **Actual backlog** requires opening pending cases, reopenings, transfers, and other adjustments. Do not label cumulative gaps as verified backlog.

## 2. Find the story — KPIs and comparison

In [ ]:
first, last = monthly.iloc[0], monthly.iloc[-1]
change_pct = (last['received'] / first['received'] - 1) * 100
ratio_change_pp = last['resolution_ratio_pct'] - first['resolution_ratio_pct']
summary = pd.DataFrame({
    'Metric': ['Complaints received', 'Complaints resolved', 'Monthly gap', 'Resolution ratio'],
    'January': [first.received, first.resolved, first.monthly_gap, first.resolution_ratio_pct],
    'June': [last.received, last.resolved, last.monthly_gap, last.resolution_ratio_pct]
})
print(f'Intake increased {change_pct:.1f}% from January to June.')
print(f'Resolution ratio changed {ratio_change_pp:+.1f} percentage points.')
display(summary.round(2))

## 3. Chart 1 — Tell the story with a headline and annotation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(monthly))
ax.plot(x, monthly['received'], marker='o', linewidth=2, label='Received')
ax.plot(x, monthly['resolved'], marker='s', linewidth=2, label='Resolved')
ax.fill_between(x, monthly['received'].to_numpy(), monthly['resolved'].to_numpy(), alpha=0.12)
ax.annotate('June net inflow: 200 cases', xy=(5, 1600), xytext=(2.1, 1670),
            arrowprops={'arrowstyle': '->'})
ax.set_xticks(x, monthly['month'].dt.strftime('%b'))
ax.set_ylabel('Number of complaints')
ax.set_title('Complaint intake is growing faster than resolution throughput')
ax.legend(); ax.grid(alpha=0.2); fig.tight_layout(); plt.show()

## 4. Chart 2 — Track service performance, not just volume

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly['month'].dt.strftime('%b'), monthly['resolution_ratio_pct'])
ax.axhline(95, linestyle='--', label='Illustrative target: 95%')
ax.set_ylim(0, 105)
ax.set_ylabel('Monthly resolution ratio (%)')
ax.set_title('Resolution throughput ratio has weakened')
ax.legend(); fig.tight_layout(); plt.show()

## 5. Where should management investigate first?
This additional **synthetic June breakdown** supports a prioritization *hypothesis*. It does not establish root cause or geographic risk by itself.

In [ ]:
zones = pd.DataFrame({
    'zone': ['North', 'Central', 'South', 'West'],
    'received': [480, 420, 390, 310],
    'resolved': [390, 380, 350, 280]
})
zones['net_inflow'] = zones['received'] - zones['resolved']
zones['resolution_ratio_pct'] = zones['resolved'] / zones['received'] * 100
assert zones['received'].sum() == 1600 and zones['resolved'].sum() == 1400
zones = zones.sort_values('net_inflow', ascending=False)
display(zones)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(zones['zone'], zones['net_inflow'])
ax.invert_yaxis(); ax.set_xlabel('June received − resolved')
ax.set_title('North zone has the largest net inflow: investigate first')
fig.tight_layout(); plt.show()

## 6. Complaint-category analysis and a concrete follow-up

In [ ]:
categories = pd.DataFrame({
    'category': ['Sewage overflow', 'Solid waste', 'Streetlights', 'Road damage'],
    'received': [520, 460, 350, 270],
    'resolved': [430, 410, 320, 240]
})
categories['net_inflow'] = categories['received'] - categories['resolved']
assert categories['received'].sum() == 1600 and categories['resolved'].sum() == 1400
display(categories.sort_values('net_inflow', ascending=False))
print('Next step: review the North-zone cases by category, age, and assigned team;')
print('do not assume that the largest overall category is concentrated in North.')

## 7. Generate a one-minute management briefing

In [ ]:
brief = f"""### Management briefing — illustrative June review
**What happened:** Complaint intake increased {change_pct:.1f}% from January to June, from 1,200 to 1,600.
June received 1,600 complaints and resolved 1,400; monthly net inflow was 200.
The June resolution throughput ratio was {last.resolution_ratio_pct:.1f}%.

**Why it matters:** Incoming demand is exceeding resolution throughput, potentially
increasing pending workloads if the pattern continues.

**Priority investigation:** North zone had the highest June net inflow (90 cases).
Review unresolved case age, category, staffing, and work-order bottlenecks.

**Action:** Assign a responsible officer, review high-age cases weekly, and test
reallocation of field capacity where operational evidence supports it.

**Measure:** Weekly intake, closures, actual pending backlog, median resolution
hours, and cases overdue against the applicable service standard.

**Illustrative goal:** Reach a monthly resolution throughput ratio of at least
95% within 30 days, subject to feasibility and service-quality safeguards.
"""
display(Markdown(brief))

## 8. Ethical and effective communication checklist
- State the dataset's source, period, coverage, and limitations.
- Distinguish observed evidence from hypotheses about causes.
- Use consistent definitions and denominators.
- Do not confuse throughput ratios with cohort resolution rates.
- Do not label a net inflow as the verified backlog.
- Use accessible chart labels and avoid misleading axes.
- Attach owners, time frames, and measurable KPIs to recommended actions.

## 9. Practice exercises
1. Add a July observation and revise the executive briefing.
2. Create a chart showing the monthly gap rather than raw complaint volume.
3. Build a zone-level ranking using both net inflow and ratio; explain when the rankings differ.
4. Draft a 100-word citizen-facing summary without operational jargon.
5. Identify additional fields required to calculate *actual* overdue backlog and time to resolution.

## 10. Reflection
**What happened? Why does it matter? What should happen next?**

**Next: Day 29 — Building an End-to-End Data Analytics Project.**

Repository: https://github.com/krishnakumarimv/30-Days-Data-Analytics-Cookbook